In [ ]:
!pip install qwen_vl_utils
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

# default: Load the model on the available device(s)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",device_map="cuda")
from PIL import Image
import torch
# default: Load the model on the available device(s)
# model = Qwen2VLForConditionalGeneration.from_pretrained(
#     # "Qwen/Qwen2-VL-32B-Instruct",
#     "Qwen/Qwen2.5-VL-3B-Instruct",
#     # device_map="cuda",
# )

# We recommend enabling flash_attention_2 for better acceleration and memory saving, especially in multi-image and video scenarios.
# model = Qwen2VLForConditionalGeneration.from_pretrained(
#     "Qwen/Qwen2-VL-7B-Instruct",
#     torch_dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     device_map="auto",
# )

# default processer
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")

# The default range for the number of visual tokens per image in the model is 4-16384. You can set min_pixels and max_pixels according to your needs, such as a token count range of 256-1280, to balance speed and memory usage.
# min_pixels = 256*28*28
# max_pixels = 1280*28*28
# processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels)


In [ ]:
image_path =  r"/content/game.jpg"
image = Image.open(image_path).convert("RGB") # Ensure the image is in RGB format

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image,
            },
            {"type": "text", "text": "i need your answer to be in this form {the_action_in_the_frame : action type,players : [{team : team color,player_num : num},{team :  team color,player_num : num}] }"},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)


In [ ]:
import os
import cv2

!ffmpeg -i /content/video.mp4 -c:v libx264 -crf 23 -preset fast /content/converted_video.mp4

# Create a folder named 'frames' if it doesn't exist
os.makedirs('frames', exist_ok=True)

# Load the video
video_path = '/content/converted_video.mp4'  # Change this if your video has a different name
cap = cv2.VideoCapture(video_path)

# Get the frame rate (frames per second)
fps = cap.get(cv2.CAP_PROP_FPS)

# Get total number of frames
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Extract one frame per second
frame_number = 0
saved_frame_count = 0

while cap.isOpened():
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ret, frame = cap.read()

    if not ret:
        break

    # Save frame as image
    frame_filename = f"frames/frame_{saved_frame_count:04d}.jpg"
    cv2.imwrite(frame_filename, frame)

    saved_frame_count += 1
    frame_number += int(fps)  # Move to the next second

cap.release()
print(f"Saved {saved_frame_count} frames in the 'frames' folder.")



folder_path = '/content/frames'

# List all files in the folder
files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

print(files)

In [ ]:
result = []
for im in files:
  image_path =  '/content/frames/' + im
  image = Image.open(image_path).convert("RGB") # Ensure the image is in RGB format

  messages = [
      {
          "role": "user",
          "content": [
              {
                  "type": "image",
                  "image": image,
              },
              {"type": "text", "text": "i need your answer to be in this form as json object '{action_type : action type,players : [{team : team color,player_num : num},{team :  team color,player_num : num}]}' if the player number is not clear make it 00"},
          ],
      }
  ]

  # Preparation for inference
  text = processor.apply_chat_template(
      messages, tokenize=False, add_generation_prompt=True
  )
  image_inputs, video_inputs = process_vision_info(messages)
  inputs = processor(
      text=[text],
      images=image_inputs,
      videos=video_inputs,
      padding=True,
      return_tensors="pt",
  )
  inputs = inputs.to("cuda")

  # Inference: Generation of the output
  generated_ids = model.generate(**inputs, max_new_tokens=128)
  generated_ids_trimmed = [
      out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
  ]
  output_text = processor.batch_decode(
      generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
  )
  print(output_text)
  result.append(str(output_text)[10:-11])


In [ ]:
import json
import re
result_cleaned = []
for idx, i in enumerate(result):  # <-- idx is the index
    cleaned = i.lstrip('n').strip()
    cleaned = cleaned.encode().decode('unicode_escape')
    cleaned = re.sub(r'"player_num": (\d+)', r'"player_num": "\1"', cleaned)

    # Fix incomplete JSON manually (if needed)
    if not cleaned.strip().endswith('}]'):
        cleaned += '\n  ]\n}'

    try:
        json_data = json.loads(cleaned)
        print(f"✅ Parsed JSON at index {idx}:")
        print(json_data)
        result_cleaned.append(json_data)
    except json.JSONDecodeError as e:
        print(f"❌ JSON decode error at index {idx}: {e}")
        print("🔍 Problematic cleaned JSON:\n", cleaned)
result_cleaned

In [ ]:
result[101] =""" {\\n  "action_type": "pass",\\n  "players": [\\n    {\\n      "team": "blue",\\n      "player_num": "1"\\n    },\\n    {\\n      "team": "blue",\\n      "player_num": "2"\\n    },\\n    {\\n      "team": "blue",\\n      "player_num": "3"\\n    },\\n    {\\n      "team": "blue",\\n      "player_num": "4"\\n    },\\n    {\\n      "team": "blue",\\n      "player_num": "5"\\n    }
                                                            ,\\n    {\\n      "team": "blue",\\n      "player_num": "2"\\n    } """

In [ ]:
print(""" n{\\n  "action_type": "pass",\\n  "players": [\\n    {\\n      "team": "blue",\\n      "player_num": "1"\\n    },\\n    {\\n      "team": "blue",\\n      "player_num": "2"\\n    },\\n    {\\n      "team": "blue",\\n      "player_num": "3"\\n    },\\n    {\\n      "team": "blue",\\n      "player_num": "4"\\n    },\\n    {\\n      "team": "blue",\\n      "player_num": "5"\\n    }
                                                            ,\\n    {\\n      "team": "blue",\\n      "player : "6"}""")

In [ ]:
type(output_text[0])

In [ ]:
str(output_text)[10:-11]

In [ ]:
with open('/content/result_cleaned.txt', 'r') as file:
    content = file.read()


In [ ]:

import ast

actual_list = ast.literal_eval(content)
print(actual_list)       # Output: [1, 2, 3, 4, 5]
print(type(actual_list))

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')  # small and fast

embeddings = model.encode(actual_list)

from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query, data, embeddings):
    query_embedding = model.encode([query])
    sims = cosine_similarity(query_embedding, embeddings)[0]
    top_idx = np.argmax(sims)
    return data[top_idx]
question = "in which indexs there is a goal?"
relevant_info = retrieve(question, actual_list, embeddings)

# Now you could send this to an LLM like ChatGPT with your question + retrieved context
print(f"Context: {relevant_info}")


In [ ]:
!pip install sentence-transformers scikit-learn ollama


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import ollama
import json

# Your data list
data = contents  # paste your list here

# Format each item for readable context
def format_entry(index, item):
    return f"Index {index}: Action = {item['action_type']}, Players = {json.dumps(item['players'])}"

entries = contents #[format_entry(i, item) for i, item in enumerate(data)]

# Embed entries
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
entry_embeddings = embed_model.encode(entries)

# Embed the question
question = "In which indexes there is a goal?"
question_embedding = embed_model.encode([question])[0]

# Retrieve top relevant entries
# similarities = cosine_similarity([question_embedding], entry_embeddings)[0]
# top_k = 20
# top_indices = np.argsort(similarities)[-top_k:][::-1]
# retrieved_context = "\n".join([entries[i] for i in top_indices])

# Build prompt for the LLM
prompt = f"""
You are analyzing structured football match data. Your job is to find the indexes in the data where a goal occurred.

Here is the data:
{entries}

Question: {question}
Answer just a list of indexes.
"""

# Run the prompt using local model (Ollama)
response = ollama.chat(
    model="mistral",
    messages=[{"role": "user", "content": prompt}]
)

print("\nModel Answer:", response['message']['content'])


In [ ]:
sub = """[{'action_type': 'goal', 'players': [{'team': 'red', 'player_num': '17'}]},
 {'action_type': 'penalty kick',
  'players': [{'team': 'red', 'player_num': '10'},
   {'team': 'blue', 'player_num': '5'}]},
 {'action_type': 'pass',
  'players': [{'team': 'red', 'player_num': '10'},
   {'team': 'blue', 'player_num': '2'}]},
 {'action_type': 'goal', 'players': [{'team': 'PSG', 'player_num': '1'}]},
 {'action_type': 'goal', 'players': [{'team': 'red', 'player_num': '1'}]},
 {'action_type': 'penalty kick',
  'players': [{'team': 'red', 'player_num': '11'},
   {'team': 'blue', 'player_num': '7'}]},
 {'action_type': 'penalty kick',
  'players': [{'team': 'red', 'player_num': '17'}]},
 {'action_type': 'pass',
  'players': [{'team': 'blue', 'player_num': '17'},
   {'team': 'blue', 'player_num': '2'}]},
 {'action_type': 'pass',
  'players': [{'team': 'red', 'player_num': '5'},
   {'team': 'blue', 'player_num': '17'}]},
 {'action_type': 'celebration',
  'players': [{'team': 'red', 'player_num': '10'}]},
 {'action_type': 'pass',
  'players': [{'team': 'red', 'player_num': '17'},
   {'team': 'blue', 'player_num': '14'}]},
 {'action_type': 'goal', 'players': [{'team': 'PSG', 'player_num': '1'}]},
 {'action_type': 'pass',
  'players': [{'team': 'blue', 'player_num': '1'},
   {'team': 'red', 'player_num': '5'}]},
 {'action_type': 'goal', 'players': [{'team': 'red', 'player_num': '1'}]},
 {'action_type': 'goal',
  'players': [{'team': 'red', 'player_num': '7'},
   {'team': 'blue', 'player_num': '1'}]},
 {'action_type': 'pass',
  'players': [{'team': 'red', 'player_num': '10'},
   {'team': 'blue', 'player_num': '2'}]},
 {'action_type': 'goal',
  'players': [{'team': 'red', 'player_num': '1'},
   {'team': 'blue', 'player_num': '2'}]},
 {'action_type': 'yellow card',
  'players': [{'team': 'PSG', 'player_num': '92'}]},
 {'action_type': 'goal',
  'players': [{'team': 'red', 'player_num': '10'},
   {'team': 'blue', 'player_num': '2'}]},
 {'action_type': 'goal', 'players': [{'team': 'red', 'player_num': '1'}]},
 {'action_type': 'penalty kick',
  'players': [{'team': 'red', 'player_num': '66'},
   {'team': 'blue', 'player_num': '1'}]},
 {'action_type': 'goal', 'players': [{'team': 'red', 'player_num': '1'}]},
 {'action_type': 'pass',
  'players': [{'team': 'blue', 'player_num': '10'},
   {'team': 'red', 'player_num': '7'}]},
 {'action_type': 'penalty kick',
  'players': [{'team': 'red', 'player_num': '1'},
   {'team': 'blue', 'player_num': '2'}]},
 {'action_type': 'clapping',
  'players': [{'team': 'black', 'player_num': '00'}]},
 {'action_type': 'celebration',
  'players': [{'team': 'PSG', 'player_num': '10'}]},
 {'action_type': 'falling', 'players': [{'team': 'red', 'player_num': '00'}]},
 {'action_type': 'penalty kick',
  'players': [{'team': 'blue', 'player_num': '14'}]},
 {'action_type': 'goal',
  'players': [{'team': 'red', 'player_num': '17'},
   {'team': 'blue', 'player_num': '10'}]},
 {'action_type': 'goal',
  'players': [{'team': 'red', 'player_num': '10'},
   {'team': 'blue', 'player_num': '7'}]},
 {'action_type': 'celebrate',
  'players': [{'team': 'red', 'player_num': '10'}]},
 {'action_type': 'goal', 'players': [{'team': 'red', 'player_num': '1'}]},
 {'action_type': 'goal', 'players': [{'team': 'blue', 'player_num': '10'}]},
 {'action_type': 'penalty kick',
  'players': [{'team': 'red', 'player_num': '11'}]},
 {'action_type': 'pass',
  'players': [{'team': 'red', 'player_num': '5'},
   {'team': 'blue', 'player_num': '10'}]},
 {'action_type': 'penalty kick',
  'players': [{'team': 'red', 'player_num': '1'},
   {'team': 'blue', 'player_num': '2'}]},
 {'action_type': 'penalty kick',
  'players': [{'team': 'red', 'player_num': '12'}]},
 {'action_type': 'goal',
  'players': [{'team': 'PSG', 'player_num': '1'},
   {'team': 'LIV', 'player_num': '00'}]},
 {'action_type': 'pass',
  'players': [{'team': 'red', 'player_num': '10'},
   {'team': 'blue', 'player_num': '7'}]},
 {'action_type': 'penalty kick',
  'players': [{'team': 'red', 'player_num': '10'},
   {'team': 'blue', 'player_num': '7'}]},
 {'action_type': 'goal', 'players': [{'team': 'red', 'player_num': '10'}]},
 {'action_type': 'goal',
  'players': [{'team': 'red', 'player_num': '1'},
   {'team': 'blue', 'player_num': '7'}]},
 {'action_type': 'penalty kick',
  'players': [{'team': 'red', 'player_num': '17'},
   {'team': 'blue', 'player_num': '6'}]},
 {'action_type': 'pass',
  'players': [{'team': 'red', 'player_num': '1'},
   {'team': 'blue', 'player_num': '2'}]},
 {'action_type': 'penalty kick',
  'players': [{'team': 'red', 'player_num': '17'}]}]"""

In [ ]:
# !pip install transformers accelerate

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v0.6"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

def ask(question, context):
    prompt = f"""You are a helpful assistant answering questions about football match data.
    considering that every dict means second
Here is the context:
{sub}

Question: {question}
Answer:"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=200)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


response = ask("how many goals?", "in indexs [1,2,3,12,13,14,15,99,100,101]")
# response = ask(
#     "In which indexes there is a goal?",
#     "Index 1: Action = goal, Players = [Player1, Player2] \nIndex 2: Action = pass, Players = [Player3] \nIndex 3: Action = goal, Players = [Player1, Player4] \nIndex 4: Action = pass, Players = [Player5]"
# )
print(response)


In [ ]:
from datasets import Dataset
import pandas as pd

# Create a Hugging Face dataset from your list of docs
data = {"title": [""] * len(custom_docs), "text": custom_docs}
dataset = Dataset.from_pandas(pd.DataFrame(data))

# Save in the format expected by RagRetriever
passages_path = "my_custom_rag_data"
dataset.save_to_disk(passages_path)


In [ ]:
from transformers import DPRContextEncoder, DPRContextEncoderTokenizer
from datasets import Dataset
import pandas as pd
import torch
import faiss
import numpy as np
import os

# Step 1: Your custom docs
custom_docs = [
    "Paris is the capital and most populous city of France.",
    "The Eiffel Tower is located in Paris, France.",
    "France is a country in Western Europe.",
]

# Step 2: Create embeddings
ctx_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
ctx_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")

with torch.no_grad():
    inputs = ctx_tokenizer(custom_docs, padding=True, truncation=True, return_tensors="pt")
    embeddings = ctx_encoder(**inputs).pooler_output  # shape (n_docs, 768)
    embedding_list = [emb.numpy() for emb in embeddings]  # convert to list of numpy arrays

# Step 3: Create HF dataset with title, text, and embeddings
dataset_dict = {
    "title": [""] * len(custom_docs),
    "text": custom_docs,
    "embeddings": embedding_list
}
dataset = Dataset.from_dict(dataset_dict)

# Step 4: Save dataset and FAISS index
passages_path = "my_custom_rag_data"
os.makedirs(passages_path, exist_ok=True)

# Save FAISS index
index = faiss.IndexFlatIP(embeddings.size(1))
index.add(embeddings.numpy())
faiss.write_index(index, os.path.join(passages_path, "index.faiss"))

# Save dataset
dataset.save_to_disk(passages_path)


In [ ]:
from transformers import RagTokenizer, RagRetriever, RagSequenceForGeneration

tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-nq")

retriever = RagRetriever.from_pretrained(
    "facebook/rag-token-nq",
    index_name="custom",
    passages_path=passages_path,
    index_path=os.path.join(passages_path, "index.faiss"),
    use_dummy_dataset=False,
)

model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-nq")
model.eval()


In [ ]:
def rag_answer(query, num_documents=5):
    # Tokenize the query
    inputs = tokenizer(query, return_tensors="pt")

    # Retrieve documents
    retriever_output = retriever.retrieve(inputs["input_ids"], n_docs=num_documents)

    # Generate the answer
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            decoder_start_token_id=model.config.pad_token_id,
            # num_return_sequences=1,
            max_length=100,
            output_scores=True,
            return_dict_in_generate=True
        )

        # Decode the generated answer
        generated_answer = tokenizer.decode(output_ids.sequences[0], skip_special_tokens=True)

        # Compute log probability of the generated tokens
        # Prepare decoder input (shifted right) to get logits
        decoder_input_ids = output_ids.sequences[:, :-1]  # exclude the last token
        labels = output_ids.sequences[:, 1:]  # exclude the first token

        # Forward pass to get logits
        outputs = model(
            input_ids=inputs["input_ids"].repeat(1, 1),  # repeat if needed
            decoder_input_ids=decoder_input_ids
        )
        logits = outputs.logits

        # Calculate log probabilities
        log_probs = F.log_softmax(logits, dim=-1)
        # Gather log probabilities of the generated tokens
        token_log_probs = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)

        # Average log probability per token (excluding padding)
        non_pad_mask = labels.ne(model.config.pad_token_id)
        avg_log_prob = (token_log_probs * non_pad_mask).sum().item() / non_pad_mask.sum().item()

        # Confidence = exp(avg log prob) to convert back from log-prob
        confidence = torch.exp(torch.tensor(avg_log_prob)).item()

    retrieved_docs = retriever_output["titles"]
    return generated_answer, retrieved_docs, confidence

question = "What is the capital of France?"
answer, retrieved_docs, confidence = rag_answer(question, num_documents=3)

print(f"Answer: {answer}")
print(f"Retrieved Documents: {retrieved_docs}")
print(f"Confidence (approx.): {confidence:.4f}")


In [ ]:
for i in list(content):
  i = str(i)

In [ ]:
!pip install transformers datasets faiss-cpu


In [ ]:
type(content)

In [ ]:
# Example data processing (convert action data into strings)
processed_data = []

for action in list(actual_list):
    action_type = action['action_type']
    players = ', '.join([f"{player['team']} player {player['player_num']}" for player in action['players']])
    description = f"{action_type.capitalize()} involving {players}"
    processed_data.append(description)

# Sample description output
print(processed_data[:5])  # Check first 5 items in the processed list


In [ ]:
from transformers import DPRContextEncoder, DPRContextEncoderTokenizer
import torch

# Load pre-trained DPR context encoder and tokenizer
ctx_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
ctx_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")

# Prepare embeddings list
embeddings = []

for description in processed_data:
    inputs = ctx_tokenizer(description, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        output = ctx_encoder(**inputs)
        embedding = output.pooler_output.squeeze().cpu().numpy()  # Extracting the embeddings
        embeddings.append(embedding)

# Check size of embeddings
print(f"Generated {len(embeddings)} embeddings.")
from datasets import Dataset

# Create a dictionary with 'text' and 'embeddings'
data = {
    "text": processed_data,
    "embeddings": embeddings
}

# Convert to Hugging Face dataset
dataset = Dataset.from_dict(data)

# Show the dataset
print(dataset)



In [ ]:
tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-nq")
model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-nq")

# Create a custom retriever.
# For simplicity, here we are simulating it by directly using the documents list as a dummy dataset.
# In practice, you would use a library like FAISS or Elasticsearch to index your dataset.

class CustomRetriever:
    def __init__(self, documents):
        self.documents = documents

    def retrieve(self, query_ids, num_return_sequences=5):
        # For simplicity, returning the first `num_return_sequences` documents
        # In a real-world scenario, this would use a search engine like FAISS
        return {"titles": self.documents[:num_return_sequences]}

retriever = CustomRetriever(processed_data)

# Step 3: Define the RAG-based question answering function
def rag_answer(query, num_documents=5):
    # Tokenize the query
    inputs = tokenizer(query, return_tensors="pt")

    # Retrieve documents using custom retriever
    retriever_output = retriever.retrieve(inputs["input_ids"], num_return_sequences=num_documents)

    # Generate the answer using the RAG model
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            decoder_start_token_id=model.config.pad_token_id,
            num_return_sequences=1,
            max_length=100,
            output_scores=True,
            return_dict_in_generate=True
        )

        # Decode the generated answer
        generated_answer = tokenizer.decode(output_ids.sequences[0], skip_special_tokens=True)

        # Compute log probability of the generated tokens
        decoder_input_ids = output_ids.sequences[:, :-1]  # exclude the last token
        labels = output_ids.sequences[:, 1:]  # exclude the first token

        # Forward pass to get logits
        outputs = model(input_ids=inputs["input_ids"], decoder_input_ids=decoder_input_ids)
        logits = outputs.logits

        # Calculate log probabilities
        log_probs = F.log_softmax(logits, dim=-1)
        token_log_probs = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)

        # Average log probability per token (excluding padding)
        non_pad_mask = labels.ne(model.config.pad_token_id)
        avg_log_prob = (token_log_probs * non_pad_mask).sum().item() / non_pad_mask.sum().item()

        # Confidence = exp(avg log prob)
        confidence = torch.exp(torch.tensor(avg_log_prob)).item()

    retrieved_docs = retriever_output["titles"]
    return generated_answer, retrieved_docs, confidence

# Step 4: Test the model with a sample query
question = "Who scored the most goals?"
answer, retrieved_docs, confidence = rag_answer(question, num_documents=3)

print(f"Answer: {answer}")
print(f"Retrieved Documents: {retrieved_docs}")
print(f"Confidence (approx.): {confidence:.4f}")


In [ ]:
from transformers import RagTokenizer, RagRetriever, RagSequenceForGeneration
import torch
import torch.nn.functional as F

# Step 1: Initialize the tokenizer, model, and retriever
tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-nq")
model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-nq")
retriever = RagRetriever.from_pretrained("facebook/rag-token-nq", index_name="legacy", use_dummy_dataset=True)

model.eval()

# Step 2: Define the question-answering function
def rag_answer(query, num_documents=5):
    # Tokenize the query
    inputs = tokenizer(query, return_tensors="pt")

    # Step 3: Retrieve documents
    retriever_output = retriever.retrieve(inputs["input_ids"], num_return_sequences=num_documents)

    # Ensure the retriever has returned something
    if retriever_output is None or retriever_output.get("titles") is None:
        return "No documents retrieved", [], 0.0

    # Retrieve the titles of the documents
    retrieved_docs = retriever_output["titles"]

    # Prepare the context input_ids (documents) to be passed into the model
    context_input_ids = retriever_output["context_input_ids"]

    # Step 4: Generate the answer using the retrieved context
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            context_input_ids=context_input_ids,
            decoder_start_token_id=model.config.pad_token_id,
            num_return_sequences=1,
            max_length=100,
            output_scores=True,
            return_dict_in_generate=True
        )

        # Decode the generated answer
        generated_answer = tokenizer.decode(output_ids.sequences[0], skip_special_tokens=True)

        # Compute log probability of the generated tokens
        decoder_input_ids = output_ids.sequences[:, :-1]  # exclude the last token
        labels = output_ids.sequences[:, 1:]  # exclude the first token

        # Forward pass to get logits
        outputs = model(input_ids=inputs["input_ids"], decoder_input_ids=decoder_input_ids)
        logits = outputs.logits

        # Calculate log probabilities
        log_probs = F.log_softmax(logits, dim=-1)
        token_log_probs = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)

        # Average log probability per token (excluding padding)
        non_pad_mask = labels.ne(model.config.pad_token_id)
        avg_log_prob = (token_log_probs * non_pad_mask).sum().item() / non_pad_mask.sum().item()

        # Confidence = exp(avg log prob)
        confidence = torch.exp(torch.tensor(avg_log_prob)).item()

    return generated_answer, retrieved_docs, confidence

# Step 5: Test the model with a sample query
question = "Who scored the most goals?"
answer, retrieved_docs, confidence = rag_answer(question, num_documents=3)

print(f"Answer: {answer}")
print(f"Retrieved Documents: {retrieved_docs}")
print(f"Confidence (approx.): {confidence:.4f}")


In [ ]:
!pip install faiss-cpu
!pip install transformers
!pip install datasets
!pip install torch


import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModel
import torch

# Load pre-trained model and tokenizer (e.g., BERT)
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Sample dataset (replace with your own data)
documents = [
    "This is the first document.",
    "This is the second document.",
    "Here is another document with more information."
]

# Encode documents using the transformer model to get embeddings
def encode_documents(documents):
    inputs = tokenizer(documents, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state.mean(dim=1)
    return embeddings

# Encode documents
doc_embeddings = encode_documents(documents)

# Convert embeddings to numpy array
doc_embeddings = doc_embeddings.cpu().numpy()

# Create FAISS index
index = faiss.IndexFlatL2(doc_embeddings.shape[1])  # L2 distance
index.add(doc_embeddings)  # Add embeddings to index


from transformers import RagTokenizer, RagRetriever, RagSequenceForGeneration

# Load RAG model and tokenizer
rag_tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-nq")
rag_retriever = RagRetriever.from_pretrained("facebook/rag-token-nq", index_name="custom", passages=documents)
rag_model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-nq", retriever=rag_retriever)

# Set up the retriever with your custom index
rag_retriever.set_passages(documents)

# Function to get response from RAG
def get_rag_response(query):
    # Tokenize the query
    inputs = rag_tokenizer(query, return_tensors="pt")

    # Generate response with RAG
    generated_ids = rag_model.generate(input_ids=inputs['input_ids'])

    # Decode the output
    return rag_tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# Test the system with a query
query = "Tell me about the second document."
response = get_rag_response(query)
print(response)




In [ ]:
import faiss
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np

# Example custom documents (replace with your dataset)
documents = [
    "This is the first document.",
    "This is the second document.",
    "Here is another document with more information."
]

# Load model and tokenizer
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Function to encode documents into embeddings
def encode_documents(documents):
    inputs = tokenizer(documents, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state.mean(dim=1)
    return embeddings

# Encode the documents
doc_embeddings = encode_documents(documents)
doc_embeddings = doc_embeddings.cpu().numpy()

# Create FAISS index
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings)

# Save the FAISS index to disk
faiss.write_index(index, "custom_faiss.index")

# Save documents as a simple text file or dataset
with open("custom_documents.txt", "w") as f:
    for doc in documents:
        f.write(f"{doc}\n")


from transformers import RagTokenizer, RagRetriever, RagSequenceForGeneration

# Load the FAISS index
index = faiss.read_index("custom_faiss.index")

# Load the documents (if you saved them as a text file)
with open("custom_documents.txt", "r") as f:
    documents = f.readlines()

# Load the RAG model and tokenizer
rag_tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-nq")
rag_retriever = RagRetriever.from_pretrained(
    "facebook/rag-token-nq",
    index_name="custom",
    passages=documents,
    index_path="custom_faiss.index"  # Point to the FAISS index file
)

# Load the RAG model for generation
rag_model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-nq", retriever=rag_retriever)

# Define a function to get a response from RAG
def get_rag_response(query):
    # Tokenize the query
    inputs = rag_tokenizer(query, return_tensors="pt")

    # Generate the response using RAG
    generated_ids = rag_model.generate(input_ids=inputs['input_ids'])

    # Decode the generated response
    return rag_tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# Example query
query = "Tell me about the second document."
response = get_rag_response(query)
print(response)


In [ ]:
import torch
import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModel
from datasets import Dataset

# Example custom documents (replace with your dataset)
documents = [
    "This is the first document.",
    "This is the second document.",
    "Here is another document with more information."
]

# Load model and tokenizer
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Function to encode documents into embeddings
def encode_documents(documents):
    inputs = tokenizer(documents, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state.mean(dim=1)
    return embeddings

# Encode the documents
doc_embeddings = encode_documents(documents)
doc_embeddings = doc_embeddings.cpu().numpy()

# Create FAISS index
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings)

# Save the FAISS index to disk
faiss.write_index(index, "custom_faiss.index")

# Save the documents into a Hugging Face Dataset
dataset = Dataset.from_dict({"text": documents})

# Save the dataset to disk
dataset.save_to_disk("custom_dataset")


from transformers import RagTokenizer, RagRetriever, RagSequenceForGeneration

# Load the saved dataset and FAISS index
dataset = Dataset.load_from_disk("custom_dataset")
index = faiss.read_index("custom_faiss.index")

# Load the RAG tokenizer and model
rag_tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-nq")
rag_retriever = RagRetriever.from_pretrained(
    "facebook/rag-token-nq",
    index_name="custom",
    passages=dataset["text"],  # Load the passages from the dataset
    index_path="custom_faiss.index"  # Point to the FAISS index file
)

# Load the RAG model
rag_model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-nq", retriever=rag_retriever)

# Function to get a response from RAG
def get_rag_response(query):
    # Tokenize the query
    inputs = rag_tokenizer(query, return_tensors="pt")

    # Generate the response using RAG
    generated_ids = rag_model.generate(input_ids=inputs['input_ids'])

    # Decode the generated response
    return rag_tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# Example query
query = "Tell me about the second document."
response = get_rag_response(query)
print(response)


In [ ]:
from transformers import DPRContextEncoder, DPRContextEncoderTokenizer
from datasets import Dataset
import torch
import faiss
import os
import numpy as np

# Load the correct tokenizer and encoder for context
ctx_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
ctx_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")

# Example custom data (replace with your actual data)
custom_docs = [
    "The capital of France is Paris.",
    "The Eiffel Tower is in Paris.",
    "France is located in Western Europe.",
    # Add more documents here
]

# Tokenize the documents
inputs = ctx_tokenizer(custom_docs, padding=True, truncation=True, max_length=512, return_tensors="pt")

# Check lengths of input data
print(f"Number of documents: {len(custom_docs)}")
print(f"Tokenized inputs: {inputs['input_ids'].shape}")

# Generate embeddings for the documents
with torch.no_grad():
    embeddings = ctx_encoder(**inputs).pooler_output

# Check the shape of embeddings
print(f"Generated embeddings shape: {embeddings.shape}")

# Convert to Hugging Face-compatible dataset format
data = {
    "text": custom_docs,  # The list of documents
    "title": ["" for _ in custom_docs],  # Title can be empty for now
    "embeddings": [emb.numpy() for emb in embeddings]  # Convert embeddings to numpy for storing
}

# Check lengths of text and embeddings
print(f"Length of text: {len(data['text'])}")
print(f"Length of embeddings: {len(data['embeddings'])}")

# Ensure the embeddings and texts are the same length
assert len(data["text"]) == len(data["embeddings"]), "Text and embeddings length mismatch!"

# Create a Hugging Face dataset
dataset = Dataset.from_dict(data)

# Save dataset and FAISS index
save_path = "my_custom_rag_data"
os.makedirs(save_path, exist_ok=True)
dataset.save_to_disk(save_path)

# FAISS index creation
index = faiss.IndexFlatIP(embeddings.shape[1])  # Use inner product for similarity search
index.add(embeddings.numpy())  # Add the embeddings to the FAISS index
faiss.write_index(index, os.path.join(save_path, "index.faiss"))

print("Dataset and FAISS index saved!")


In [ ]:
from transformers import DPRContextEncoder, DPRContextEncoderTokenizer
from datasets import Dataset
import torch
import faiss
import numpy as np
import pandas as pd
import os

# Load DPR encoder
ctx_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
ctx_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")

# Embed documents
inputs = ctx_tokenizer(content, padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    embeddings = ctx_encoder(**inputs).pooler_output

# Convert to HF-compatible dataset
data = {
    "text": content,
    "title": ["" for _ in content],  # required but can be empty
    "embeddings": [emb.numpy() for emb in embeddings]
}
dataset = Dataset.from_dict(data)

# Save dataset and FAISS index
save_path = "my_custom_rag_data"
os.makedirs(save_path, exist_ok=True)
dataset.save_to_disk(save_path)

# Save FAISS index
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.numpy())
faiss.write_index(index, os.path.join(save_path, "index.faiss"))


In [ ]:
question = "What is the capital of France?"
answer, retrieved_docs, confidence = rag_answer(question, num_documents=10)

print(f"Answer: {answer}")
print(f"Retrieved Documents: {retrieved_docs}")
print(f"Confidence (approx.): {confidence:.4f}")


In [ ]:
def rag_answer(query, num_documents=3):
    inputs = tokenizer(query, return_tensors="pt")
    model.set_retriever(retriever)

    with torch.no_grad():
        # Note: return_dict_in_generate=False (default)
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            num_return_sequences=1,
            decoder_start_token_id=model.config.pad_token_id,
            num_beams=4,
            max_length=100,
            output_scores=True,
            n_docs=num_documents,
            do_deduplication=False
        )

        answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        # Confidence estimation: re-run forward pass
        decoder_input_ids = output_ids[:, :-1]
        labels = output_ids[:, 1:]

        outputs = model(
            input_ids=inputs["input_ids"].repeat(1, 1),
            decoder_input_ids=decoder_input_ids
        )
        logits = outputs.logits
        log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
        token_log_probs = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)

        non_pad_mask = labels.ne(model.config.pad_token_id)
        avg_log_prob = (token_log_probs * non_pad_mask).sum().item() / non_pad_mask.sum().item()
        confidence = torch.exp(torch.tensor(avg_log_prob)).item()

    return answer, None, confidence


In [ ]:
retriever = RagRetriever.from_pretrained(
    "facebook/rag-token-nq",
    index_name="custom",
    passages_path=passages_path,
    index_path=os.path.join(passages_path, "index.faiss"),
    use_dummy_dataset=False,
)


In [ ]:
from transformers import (
    RagTokenizer, RagRetriever, RagSequenceForGeneration,
    DPRContextEncoder, DPRContextEncoderTokenizer
)
import torch
import faiss
import os
import numpy as np

# Step 1: Your custom documents
custom_docs = [
    "Paris is the capital and most populous city of France.",
    "The Eiffel Tower is located in Paris, France.",
    "France is a country in Western Europe.",
]

# Step 2: Encode documents into embeddings using DPR encoder
ctx_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
ctx_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")

with torch.no_grad():
    inputs = ctx_tokenizer(custom_docs, padding=True, truncation=True, return_tensors="pt")
    embeddings = ctx_encoder(**inputs).pooler_output  # (n_docs, hidden_size)

# Step 3: Create FAISS index
index = faiss.IndexFlatIP(embeddings.size(1))  # Use inner product similarity
index.add(embeddings.numpy())

# Step 4: Save passages and index to local directory
passages_dir = "my_custom_rag_data"
os.makedirs(passages_dir, exist_ok=True)
faiss.write_index(index, os.path.join(passages_dir, "index.faiss"))
np.save(os.path.join(passages_dir, "passages.npy"), np.array(custom_docs))

# Step 5: Initialize tokenizer, retriever (custom), and model
tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-nq")

retriever = RagRetriever.from_pretrained(
    "facebook/rag-token-nq",
    index_name="custom",
    passages_path=passages_dir,
    index_path=os.path.join(passages_dir, "index.faiss"),
    use_dummy_dataset=False,
)

model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-nq")
model.eval()




In [ ]:
!pip install datasets
!pip install faiss-cpu

from transformers import RagTokenizer, RagRetriever, RagSequenceForGeneration
import torch
import torch.nn.functional as F

# Initialize the tokenizer and RAG model
tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-nq")
retriever = RagRetriever.from_pretrained("facebook/rag-token-nq", index_name="legacy", use_dummy_dataset=True)
model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-nq")
model.eval()

def rag_answer(query, num_documents=5):
    # Tokenize the query
    inputs = tokenizer(query, return_tensors="pt")

    # Retrieve documents
    retriever_output = retriever.retrieve(inputs["input_ids"], num_return_sequences=num_documents)

    # Generate the answer
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            decoder_start_token_id=model.config.pad_token_id,
            num_return_sequences=1,
            max_length=100,
            output_scores=True,
            return_dict_in_generate=True
        )

        # Decode the generated answer
        generated_answer = tokenizer.decode(output_ids.sequences[0], skip_special_tokens=True)

        # Compute log probability of the generated tokens
        # Prepare decoder input (shifted right) to get logits
        decoder_input_ids = output_ids.sequences[:, :-1]  # exclude the last token
        labels = output_ids.sequences[:, 1:]  # exclude the first token

        # Forward pass to get logits
        outputs = model(
            input_ids=inputs["input_ids"].repeat(1, 1),  # repeat if needed
            decoder_input_ids=decoder_input_ids
        )
        logits = outputs.logits

        # Calculate log probabilities
        log_probs = F.log_softmax(logits, dim=-1)
        # Gather log probabilities of the generated tokens
        token_log_probs = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)

        # Average log probability per token (excluding padding)
        non_pad_mask = labels.ne(model.config.pad_token_id)
        avg_log_prob = (token_log_probs * non_pad_mask).sum().item() / non_pad_mask.sum().item()

        # Confidence = exp(avg log prob) to convert back from log-prob
        confidence = torch.exp(torch.tensor(avg_log_prob)).item()

    retrieved_docs = retriever_output["titles"]
    return generated_answer, retrieved_docs, confidence

# Example usage
question = "What is the capital of France?"
answer, retrieved_docs, confidence = rag_answer(question, num_documents=3)

print(f"Answer: {answer}")
print(f"Retrieved Documents: {retrieved_docs}")
print(f"Confidence (approx.): {confidence:.4f}")


In [ ]:
content

In [ ]:
test = """['```json\n{\n  "action_type": "goal",\n  "players": [\n    {\n      "team": "red",\n      "player_num": 17\n    }\n  ]\n}\n```']
"""
print(test[10:-10])

In [ ]:
!pip install pytube
!pip install opencv-python
!pip install yt-dlp

In [ ]:
import yt_dlp

def download_youtube_video(url, save_path="video.mp4"):
    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]',
        'outtmpl': save_path,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    return save_path

# Example usage
video_path = download_youtube_video("https://youtu.be/pBNkcCWbjTk?si=Js1ObZcYiynSR_Ai")
print(f"Downloaded: {video_path}")


In [ ]:
import cv2

video_path = "video.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Cannot open video file.")
else:
    print("Video successfully opened.")

cap.release()


In [ ]:
import cv2

video_path = "video.mp4"
cap = cv2.VideoCapture(video_path)

if cap.isOpened():
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"FPS: {fps}, Total Frames: {total_frames}")
else:
    print("Error: Cannot open video file.")

cap.release()

In [ ]:
!apt-get install -y ffmpeg  # Ensure ffmpeg is installed
!ffmpeg -i video.mp4 -vcodec libx264 -preset ultrafast converted.mp4

In [ ]:
import cv2

def extract_frames(video_path, frame_interval=1):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error: Cannot open video file.")
        return []

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Video FPS: {fps}, Total Frames: {total_frames}")  # Debugging info

    frame_count = 0
    frames = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("End of video reached.")
            break

        if frame_count % int(fps * frame_interval) == 0:
            timestamp = frame_count / fps
            frames.append((frame, timestamp))
            print(f"Extracted frame at {timestamp:.2f}s")  # Debugging output

        frame_count += 1

    cap.release()
    print(f"Total frames extracted: {len(frames)}")
    return frames

frames = extract_frames("converted.mp4")


In [ ]:
import os
import cv2
import yt_dlp
import numpy as np

# ------------------------ Step 1: Download YouTube Video ------------------------
def download_youtube_video(url, output_filename="video.mp4"):
    ydl_opts = {
        'format': 'bestvideo+bestaudio/best',
        'outtmpl': output_filename,
        'quiet': True
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    return output_filename

# ------------------------ Step 2: Convert Video for Compatibility ------------------------
def convert_video(input_path, output_path="converted.mp4"):
    os.system(f"ffmpeg -i {input_path} -vcodec libx264 -preset ultrafast {output_path}")
    return output_path

# ------------------------ Step 3: Extract Frames ------------------------
def extract_frames(video_path, output_folder="frames", frame_interval=50):
    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"🎥 Video FPS: {fps}, Total Frames: {total_frames}")

    frame_count = 0
    extracted_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break  # End of video

        if frame_count % frame_interval == 0:
            timestamp = frame_count / fps  # Time in seconds
            frame_filename = f"{output_folder}/frame_{frame_count:05d}_t{timestamp:.2f}.jpg"
            cv2.imwrite(frame_filename, frame)
            extracted_count += 1

        frame_count += 1

    cap.release()
    print(f"✅ Total frames extracted: {extracted_count}")

# ------------------------ Step 4: Full Pipeline Execution ------------------------
def process_video(youtube_url):
    print("📥 Downloading video...")
    raw_video = download_youtube_video(youtube_url)

    print("🎬 Converting video format...")
    converted_video = convert_video(raw_video)

    print("🖼️ Extracting frames...")
    extract_frames(converted_video, frame_interval=50)  # Adjust interval as needed

    print("🚀 Processing complete!")


In [ ]:
extract_frames('/content/converted.mp4', frame_interval=50)

In [ ]:

# ------------------------ Run the Pipeline ------------------------
youtube_url = "https://youtu.be/pBNkcCWbjTk?si=Js1ObZcYiynSR_Ai"  # Replace with actual video URL
process_video(youtube_url)


In [ ]:
!pip install transformers accelerate torch torchvision

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration, BertTokenizerFast
import torch
from PIL import Image
import os
import json

# Load BLIP-2 Model
processor = BlipProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b").to("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
def analyze_frames(frame_folder="frames", output_file="descriptions.txt"):
    frames = sorted(os.listdir(frame_folder))  # Sort frames in order
    results = []

    for frame in frames:
        frame_path = os.path.join(frame_folder, frame)
        image = Image.open(frame_path).convert("RGB")

        # Prepare input for BLIP-2
        inputs = processor(images=image, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

        # Generate caption
        with torch.no_grad():
            caption = model.generate(**inputs)
            description = processor.decode(caption[0], skip_special_tokens=True)

        # Extract timestamp from filename
        timestamp = frame.split("_t")[1].replace(".jpg", "")
        results.append(f"[{timestamp}s] {description}")
        print(f"📝 {timestamp}s → {description}")

    # Save results
    with open(output_file, "w") as f:
        f.write("\n".join(results))

    print(f"✅ Saved descriptions to {output_file}")

# Run the frame analysis
analyze_frames()